# masarch / agentorch 接口 smoke test

这个 notebook 用来在 Jupyter 里直接导入 `agentorch`，对当前公开接口做一轮离线优先、可重复的 smoke test。

覆盖思路：
- 校验分发名 `masarch` 与导入名 `agentorch` 的映射。
- 批量校验顶层公开导出是否存在。
- 对核心接口簇做真实构造和执行，而不只是 `hasattr()`。
- 真实模型链路只在环境变量齐全时执行，不默认读取 `.env`。

说明：
- Notebook 环境里统一使用 `await`，避免 `run_sync()` / `close()` 在事件循环中报错。
- 每次 `Agent.run(...)` 都显式传入 `thread_id`。
- 默认不会触发真实外部 API；只有最后的可选单元才会尝试读取当前 shell 中已存在的 `OPENAI_*` 环境变量。

In [ ]:
from __future__ import annotations

import asyncio
import importlib.metadata as metadata
import json
import os
import random
import tempfile
from pathlib import Path
from uuid import uuid4

from pydantic import BaseModel

import agentorch
from agentorch.agents import SharedNote, TaskArtifact
from agentorch.config import MemoryConfig, ModelConfig, RuntimeConfig, SkillCatalogConfig, SkillRoutingConfig, validate_supported_python
from agentorch.core import Message, ModelRequest, ModelResponse, UsageInfo
from agentorch.evolution import EvolutionAlgorithm
from agentorch.knowledge import Document, DocumentChunk, KnowledgeAsset, RetrievalQuery
from agentorch.models.base import BaseModelAdapter
from agentorch.models.embedding import EmbeddingCapableModelAdapter
from agentorch.models.media import ImageGenerationCapableModelAdapter, ImageGenerationResult, VideoAnalysisCapableModelAdapter
from agentorch.models.speech import SpeechCapableModelAdapter, SpeechSynthesisResult
from agentorch.reasoning import BaseReasoningFramework, ReactConfig, ReasoningResult
from agentorch.sandbox import SandboxManager, SandboxPolicy
from agentorch.tools.base import FunctionTool
from agentorch.workflow import Node

ROOT = Path.cwd().resolve()
TMP_ROOT = ROOT / ".tmp_notebook_smoke"
TMP_ROOT.mkdir(exist_ok=True)


def banner(title: str) -> None:
    print(f"\n{'=' * 24} {title} {'=' * 24}")


def ensure(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(message)


def dump_sample(value, *, max_chars: int = 600) -> None:
    text = json.dumps(value, ensure_ascii=False, indent=2, default=str)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n...<truncated>..."
    print(text)


def unique_name(prefix: str) -> str:
    return f"{prefix}_{uuid4().hex[:8]}"


class ValueModel(BaseModel):
    value: str


class AddInput(BaseModel):
    a: int
    b: int


class EchoInput(BaseModel):
    text: str


class DummyModel(BaseModelAdapter):
    def __init__(self, *, name: str = "dummy-model", reply: str = "ok") -> None:
        self.config = {"provider": "dummy", "api_key": "sk-dummy-notebook", "model": name}
        self.reply = reply
        self.closed = False
        self.requests: list[ModelRequest] = []

    async def generate(self, request: ModelRequest) -> ModelResponse:
        self.requests.append(request)
        return ModelResponse(
            message=Message(role="assistant", content=self.reply),
            content=self.reply,
            finish_reason="stop",
            usage=UsageInfo(total_tokens=5),
        )

    async def aclose(self) -> None:
        self.closed = True


class RecordingPromptModel(BaseModelAdapter):
    def __init__(self, reply: str = "recorded") -> None:
        self.config = {"provider": "dummy", "api_key": "sk-dummy-record", "model": "prompt-recorder"}
        self.reply = reply
        self.system_prompts: list[str] = []

    async def generate(self, request: ModelRequest) -> ModelResponse:
        system_text = next((message.content for message in request.messages if message.role == "system"), "")
        self.system_prompts.append(system_text)
        return ModelResponse(
            message=Message(role="assistant", content=self.reply),
            content=self.reply,
            finish_reason="stop",
            usage=UsageInfo(total_tokens=1),
        )


class DemoReasoningFramework(BaseReasoningFramework):
    async def execute(self, runtime, context) -> ReasoningResult:
        session = self.initialize(context)
        session.state["final_output"] = "demo-reasoning"
        session.metadata["kind"] = self.config.kind.value
        return self.finalize(session)


class DemoEvolutionAlgorithm(EvolutionAlgorithm):
    def __init__(self, marker: str = "demo-evolution") -> None:
        self.marker = marker

    async def evolve(self, context, *, tasks=None, initial_population=None):
        genome = agentorch.Genome(id="demo-genome", genes={"marker": self.marker})
        evaluation = agentorch.EvaluationResult(genome_id=genome.id, fitness=9.0, metrics={"tasks": float(len(tasks or []))})
        return agentorch.EvolutionResult(best_genome=genome, best_evaluation=evaluation)


class StubSpeechImageVideoModel(
    SpeechCapableModelAdapter,
    ImageGenerationCapableModelAdapter,
    VideoAnalysisCapableModelAdapter,
):
    def __init__(self) -> None:
        self.config = {
            "provider": "stub-media",
            "model": "stub-media-model",
            "speech_format": "mp3",
            "image_timeout": 30.0,
            "image_model": "stub-image-model",
            "video_model": "stub-video-model",
        }

    async def generate(self, request: ModelRequest) -> ModelResponse:
        return ModelResponse(
            message=Message(role="assistant", content="stub-media-chat"),
            content="stub-media-chat",
            finish_reason="stop",
            usage=UsageInfo(total_tokens=1),
        )

    async def synthesize_speech(
        self,
        text: str,
        *,
        voice: str | None = None,
        response_format: str | None = None,
        speed: float | int | str | None = None,
        speech_model: str | None = None,
        output_path: str | Path | None = None,
    ) -> SpeechSynthesisResult:
        target = Path(output_path or (TMP_ROOT / "stub_audio.mp3")).resolve()
        target.parent.mkdir(parents=True, exist_ok=True)
        payload = f"FAKE-MP3::{text}".encode("utf-8")
        target.write_bytes(payload)
        return SpeechSynthesisResult(
            output_path=str(target),
            response_format=(response_format or "mp3"),
            content_type="audio/mpeg",
            bytes_written=len(payload),
            voice=voice or "alloy",
            speed=float(speed if speed is not None else 1.0),
            model=speech_model or "stub-speech-model",
        )

    async def generate_image(
        self,
        prompt: str,
        *,
        aspect_ratio: str | None = None,
        image_size: str | None = None,
        image_model: str | None = None,
        output_path: str | Path | None = None,
    ) -> ImageGenerationResult:
        target = Path(output_path or (TMP_ROOT / "stub_image.png")).resolve()
        target.parent.mkdir(parents=True, exist_ok=True)
        png_bytes = bytes.fromhex(
            "89504E470D0A1A0A0000000D4948445200000001000000010802000000907753DE"
            "0000000C49444154789C63606060000000040001F61738550000000049454E44AE426082"
        )
        target.write_bytes(png_bytes)
        return ImageGenerationResult(
            output_path=str(target),
            mime_type="image/png",
            bytes_written=len(png_bytes),
            model=image_model or "stub-image-model",
            proxy_mode="stub",
        )

    async def analyze_video(
        self,
        *,
        prompt: str,
        video_path: str | Path | None = None,
        video_url: str | None = None,
        mime_type: str | None = None,
        model: str | None = None,
        max_tokens: int | None = None,
        temperature: float | None = None,
    ) -> ModelResponse:
        source = str(video_path or video_url or "unknown")
        content = f"video analysis ok :: {source} :: {prompt}"
        raw = type("RawVideoResponse", (), {"model": model or "stub-video-model"})()
        return ModelResponse(
            message=Message(role="assistant", content=content),
            content=content,
            finish_reason="stop",
            usage=UsageInfo(total_tokens=3),
            raw=raw,
        )


validate_supported_python()
banner("Environment")
print(f"cwd={ROOT}")
print(f"python={os.sys.version.split()[0]}")
print(f"agentorch file={agentorch.__file__}")

In [ ]:
banner("Package Identity")
dist_version = None
try:
    dist_version = metadata.version("masarch")
except metadata.PackageNotFoundError:
    dist_version = None

identity = {
    "distribution_name": "masarch",
    "distribution_version": dist_version,
    "import_name": "agentorch",
    "import_file": agentorch.__file__,
}
dump_sample(identity)
print("说明：如果 distribution_version 为 null，代表当前是在源码目录直接导入，而不是从已安装 wheel 导入。")

In [ ]:
banner("Top-level Export Surface")
stable_symbols = sorted(set(agentorch.STABLE_API_SYMBOLS))
compat_symbols = sorted(set(agentorch.COMPAT_API_SYMBOLS))

missing_stable = [name for name in stable_symbols if not hasattr(agentorch, name)]
missing_compat = [name for name in compat_symbols if not hasattr(agentorch, name)]

ensure(not missing_stable, f"缺失稳定导出: {missing_stable}")
ensure(not missing_compat, f"缺失兼容导出: {missing_compat}")
ensure(set(agentorch.__all__) == set(stable_symbols), "__all__ 与 STABLE_API_SYMBOLS 不一致")

summary = {
    "stable_count": len(stable_symbols),
    "compat_count": len(compat_symbols),
    "stable_sample": stable_symbols[:20],
    "compat_sample": compat_symbols[:20],
}
dump_sample(summary)
print("顶层公开导出存在性校验通过。")

In [ ]:
banner("Config and Parser APIs")

model_config = ModelConfig.from_any("demo-model", provider="openai_http", max_tokens=128)
agent_config = RuntimeConfig.agent(
    system_prompt="Use evidence.",
    rag="classic",
    reasoning="react",
    default_knowledge_scope=["demo"],
)
workflow_config = RuntimeConfig.workflow(reasoning="plan_execute", max_steps=12)
catalog_config = SkillCatalogConfig.from_any({"discovery_roots": [".skills"], "selection_mode": "hybrid"})
routing_config = SkillRoutingConfig.from_any("progressive")

parsed_json = await agentorch.JSONParser().parse("```json\n{\"name\": \"agentorch\"}\n```")
parsed_list = await agentorch.ListParser().parse("- alpha\n- beta")
parsed_kv = await agentorch.KeyValueParser().parse("x: 1\ny=2")
parsed_pydantic = await agentorch.PydanticParser(ValueModel).parse("hello notebook")
parser = agentorch.parser_chain(agentorch.JSONParser(), agentorch.TextParser())
parsed_chain = await parser.parse('{"ok": true}')
prompt = agentorch.format_prompt("Summarize the result", agentorch.TextParser())

ensure(model_config.model == "demo-model", "ModelConfig.from_any 失败")
ensure(model_config.provider == "openai_http", "provider 解析失败")
ensure(agent_config.reasoning_strategy is not None and agent_config.reasoning_strategy.kind.value == "react", "agent runtime reasoning 配置失败")
ensure(agent_config.rag_strategy is not None and agent_config.rag_strategy.mode == "classic", "agent runtime rag 配置失败")
ensure(workflow_config.retrieval_mode.value == "explicit_step", "workflow runtime retrieval_mode 错误")
ensure(catalog_config.discovery_roots == [".skills"], "SkillCatalogConfig 错误")
ensure(routing_config.selection_mode == "model", "SkillRoutingConfig 字符串快捷方式异常")
ensure(parsed_json == {"name": "agentorch"}, "parse_json 失败")
ensure(parsed_list == ["alpha", "beta"], "parse_list 失败")
ensure(parsed_kv == {"x": "1", "y": "2"}, "parse_key_values 失败")
ensure(parsed_pydantic.value == "hello notebook", "parse_pydantic 失败")
ensure(parsed_chain == {"ok": True}, "parser_chain 失败")
ensure("Output format" in prompt, "format_prompt 未注入格式说明")

dump_sample(
    {
        "model_config": model_config.model_dump(),
        "agent_runtime_config": agent_config.model_dump(mode="json"),
        "workflow_runtime_config": workflow_config.model_dump(mode="json"),
        "parsed_examples": {
            "json": parsed_json,
            "list": parsed_list,
            "key_values": parsed_kv,
            "pydantic": parsed_pydantic.model_dump(),
            "chain": parsed_chain,
        },
    }
)
print("配置与解析接口 smoke test 通过。")

In [ ]:
banner("Knowledge and Memory APIs")

tmp_text_path = TMP_ROOT / "knowledge_note.txt"
tmp_text_path.write_text("framework test content for retrieval", encoding="utf-8")

retriever = agentorch.create_retriever(
    "keyword",
    chunks=[
        DocumentChunk(
            id="chunk-1",
            document_id="doc-1",
            text="agent orchestration with retrieval",
            metadata={"scopes": ["papers"]},
        )
    ],
)
retrieval = await retriever.retrieve(RetrievalQuery(query="retrieval", top_k=1, scopes=["papers"]))

chunker = agentorch.create_chunking_strategy("fixed_token_chunk", chunk_size=2, chunk_overlap=0)
chunks = await chunker.chunk([Document(id="doc-2", text="one two three four")])

asset = KnowledgeAsset.from_path(tmp_text_path, scope_tags=["demo"])
adapter = agentorch.create_document_adapter("text")
parsed_document = await adapter.parse(asset)

embedding_name = unique_name("embedding")
reranker_name = unique_name("reranker")
agentorch.register_embedding_provider(embedding_name, factory=lambda dimensions=3: {"dimensions": dimensions})
agentorch.register_reranker(reranker_name, factory=lambda top_k=1: {"top_k": top_k})

kb = agentorch.InMemoryKnowledgeBase()
await kb.ingest([
    Document(id="memo-1", text="agentorch keeps runtime concerns explicit", metadata={"title": "memo-1", "scopes": ["demo"]})
])
kb_retriever = kb.get_retriever()
kb_hits = await kb_retriever.retrieve(RetrievalQuery(query="runtime", top_k=1, scopes=["demo"]))

indexed_kb = await agentorch.IndexedKnowledgeBase.acreate(
    documents=[Document(id="doc-indexed", text="indexed knowledge base retrieval demo", metadata={"title": "Indexed Demo", "scopes": ["indexed"]})]
)
indexed_report = await indexed_kb.get_retriever().retrieve_report(
    agentorch.RetrievalIntent.from_question("retrieval demo", knowledge_scope=["indexed"], max_documents=2)
)

memory_config = MemoryConfig(
    checkpoint_path=TMP_ROOT / "checkpoints.db",
    record_path=TMP_ROOT / "records.db",
)
manager = agentorch.MemoryManager.compose(
    "session_memory",
    "thread_summary_memory",
    "agent_local_memory",
    "workspace_memory",
    "shared_note_memory",
    "record_memory",
    "collective_memory",
    config=memory_config,
)

await manager.append_message("thread-1", Message(role="user", content="hello framework"))
messages = await manager.get_thread_messages("thread-1")
summary = await manager.summarize_thread("thread-1")
await manager.append_agent_memory("thread-1", "coder", {"goal": "fix tests"})
agent_memory = await manager.get_agent_memory("thread-1", "coder")

artifact = TaskArtifact(name="draft", content="artifact-content")
workspace_record = await manager.write_workspace_record("thread-1", task_id="task-1", owner_agent="coder", artifact=artifact)
workspace_records = await manager.read_workspace_records("thread-1")

note = SharedNote(note_id="note-1", task_id="task-1", author_agent="coder", content="candidate insight", metadata={"collective_candidate": True})
await manager.add_shared_note("thread-1", note)
notes = await manager.get_shared_notes("thread-1")
candidates = await manager.collect_candidate_notes("thread-1", task_id="task-1")

collective_id = await manager.promote_collective_memory(
    thread_id="thread-1",
    kind="lesson",
    content="validated answer",
    tags=["memory"],
    source_agents=["coder"],
)
collective = await manager.search_collective_memory(query="validated", thread_id="thread-1")
validated = await manager.validate_collective_memory(collective_id)
deprecated = await manager.deprecate_collective_memory(collective_id)
await manager.checkpoint("thread-1", "cp-1", {"step": 1})
checkpoint = await manager.load_checkpoint("thread-1", "cp-1")

ensure(retrieval[0].chunk.document_id == "doc-1", "keyword retriever 失败")
ensure([chunk.text for chunk in chunks] == ["one two", "three four"], "chunker 结果不符合预期")
ensure(parsed_document.title == "knowledge_note.txt", "document adapter 失败")
ensure(agentorch.create_embedding_provider(embedding_name, dimensions=6)["dimensions"] == 6, "embedding provider 注册失败")
ensure(agentorch.create_reranker(reranker_name, top_k=4)["top_k"] == 4, "reranker 注册失败")
ensure(kb_hits and kb_hits[0].chunk.document_id == "memo-1", "InMemoryKnowledgeBase 检索失败")
ensure(indexed_report.evidence, "IndexedKnowledgeBase retrieval_report 为空")
ensure([message.content for message in messages] == ["hello framework"], "memory append/get 失败")
ensure("hello framework" in summary, "memory summary 失败")
ensure(agent_memory == [{"goal": "fix tests"}], "agent memory 失败")
ensure(workspace_record.name == "draft", "workspace artifact 写入失败")
ensure(workspace_records[0].content == "artifact-content", "workspace artifact 读取失败")
ensure(notes[0].content == "candidate insight", "shared note 失败")
ensure(candidates[0].note_id == "note-1", "candidate note 收集失败")
ensure(collective and collective[0]["kind"] == "lesson", "collective memory 检索失败")
ensure(validated is not None and validated["status"] == "validated", "validate_collective_memory 失败")
ensure(deprecated is not None and deprecated["status"] == "deprecated", "deprecate_collective_memory 失败")
ensure(checkpoint == {"step": 1}, "checkpoint 失败")

dump_sample(
    {
        "retrieval_hit": retrieval[0].model_dump(mode="json"),
        "parsed_document": parsed_document.model_dump(mode="json"),
        "indexed_report_summary": indexed_report.summary,
        "memory_mechanisms": manager.list_mechanisms(),
        "workspace_record": workspace_record.model_dump(mode="json"),
        "collective_memory_hit": collective[0],
    }
)
print("知识库与记忆接口 smoke test 通过。")

In [ ]:
banner("Registry APIs")

provider_name = unique_name("provider")
reasoning_name = unique_name("reasoning")
evolution_name = unique_name("evolution")
memory_backend_name = unique_name("memory_backend")

agentorch.register_model_provider(provider_name, lambda config: DummyModel(name=config.model or "missing", reply="provider-ok"))
agentorch.register_reasoning_framework(reasoning_name, None, factory=lambda max_steps=2: DemoReasoningFramework(ReactConfig(max_steps=max_steps)))
agentorch.register_evolution_algorithm(evolution_name, None, factory=lambda marker="custom": DemoEvolutionAlgorithm(marker=marker))
agentorch.register_memory_backend(memory_backend_name, factory=lambda marker="ok": {"marker": marker})

provider_model = agentorch.create_model_adapter({"provider": provider_name, "model": "custom-model"})
reasoning_framework = agentorch.create_reasoning_framework(reasoning_name, max_steps=5)
evolution_algorithm = agentorch.create_evolution_algorithm(evolution_name, marker="explicit")
custom_memory_backend = agentorch.create_memory_backend(memory_backend_name, marker="custom")

ensure(provider_name in agentorch.list_model_providers(), "model provider 注册失败")
ensure(isinstance(provider_model, DummyModel), "自定义 model provider 未返回 DummyModel")
ensure(reasoning_name in agentorch.list_reasoning_frameworks(), "reasoning framework 注册失败")
ensure(isinstance(reasoning_framework, DemoReasoningFramework), "custom reasoning framework 类型错误")
ensure(reasoning_framework.config.max_steps == 5, "custom reasoning framework 参数透传失败")
ensure(evolution_name in agentorch.list_evolution_algorithms(), "evolution algorithm 注册失败")
ensure(isinstance(evolution_algorithm, DemoEvolutionAlgorithm), "custom evolution algorithm 类型错误")
ensure(evolution_algorithm.marker == "explicit", "custom evolution algorithm 参数透传失败")
ensure(custom_memory_backend == {"marker": "custom"}, "custom memory backend 失败")

dump_sample(
    {
        "model_providers_tail": sorted(agentorch.list_model_providers())[-5:],
        "reasoning_frameworks_tail": sorted(agentorch.list_reasoning_frameworks())[-5:],
        "evolution_algorithms_tail": sorted(agentorch.list_evolution_algorithms())[-5:],
        "custom_provider_model_config": provider_model.config,
        "custom_memory_backend": custom_memory_backend,
    }
)
print("注册表接口 smoke test 通过。")

In [ ]:
banner("Tool APIs")

tool_workspace = TMP_ROOT / "tool_workspace"
tool_workspace.mkdir(exist_ok=True)
sample_file = tool_workspace / "sample.txt"
sample_file.write_text("line1\nline2\nline3", encoding="utf-8")

@agentorch.tool(description="Add two integers")
def add_numbers(payload: AddInput):
    return {"sum": payload.a + payload.b}


async def echo_text(payload: EchoInput):
    return {"echo": payload.text}


custom_registry = agentorch.ToolRegistry.empty().register_many(add_numbers)
echo_tool = FunctionTool(
    name="echo_text",
    description="Echo text",
    input_model=EchoInput,
    func=echo_text,
)
custom_registry.extend(agentorch.ToolRegistry.from_tools(echo_tool))

fs_registry = agentorch.ToolRegistry.with_bundles(
    workspace_root=tool_workspace,
    include_filesystem=True,
    include_execution=False,
    include_git=False,
)

sandbox = SandboxManager(
    policy=SandboxPolicy(
        allowed_paths=[tool_workspace],
        command_allowlist=["powershell", "cmd", "python", "py"],
        allow_shell=False,
    )
)
exec_registry = agentorch.ToolRegistry.with_bundles(
    workspace_root=tool_workspace,
    sandbox=sandbox,
    include_filesystem=False,
    include_execution=True,
    include_git=False,
)

git_registry = agentorch.ToolRegistry.with_bundles(
    workspace_root=ROOT,
    include_filesystem=False,
    include_execution=False,
    include_git=True,
)

media_model = StubSpeechImageVideoModel()
media_registry = agentorch.ToolRegistry.with_bundles(
    workspace_root=tool_workspace,
    include_filesystem=False,
    include_execution=False,
    include_git=False,
    include_media=True,
    model=media_model,
)

video_path = tool_workspace / "sample.mp4"
video_path.write_bytes(b"FAKE-MP4")

add_result = await custom_registry.execute("add_numbers", {"a": 2, "b": 3})
echo_result = await custom_registry.execute("echo_text", {"text": "hello"})
list_result = await fs_registry.execute("list_directory", {"path": ".", "include_size": True})
read_result = await fs_registry.execute("read_file", {"path": "sample.txt", "include_line_numbers": True})
write_result = await fs_registry.execute("write_file", {"path": "written.txt", "content": "written by notebook"})
command_result = await exec_registry.execute("run_command", {"argv": ["powershell", "-Command", "Write-Output tool-ok"], "workdir": str(tool_workspace)})
git_status_result = await git_registry.execute("git_status", {"path": "."})
git_commits_result = await git_registry.execute("git_recent_commits", {"path": ".", "limit": 3})
git_diff_result = await git_registry.execute("git_diff_summary", {"path": "."})
tts_result = await media_registry.execute("text_to_speech", {"text": "hello media"})
image_result = await media_registry.execute("generate_image", {"prompt": "draw a single pixel test image"})
video_result = await media_registry.execute("analyze_video", {"question": "what is in this file", "video_path": "sample.mp4", "save_outputs": True})

ensure(add_result.data["sum"] == 5, "custom add tool 失败")
ensure(echo_result.data["echo"] == "hello", "custom echo tool 失败")
ensure(any(entry["path"] == "sample.txt" for entry in list_result.data["entries"]), "list_directory 未看到 sample.txt")
ensure("1: line1" in read_result.data["content"], "read_file 行号输出失败")
ensure((tool_workspace / "written.txt").exists(), "write_file 未写出文件")
ensure(command_result.data["exit_code"] == 0 and "tool-ok" in command_result.data["stdout"], "run_command 失败")
ensure("branch" in git_status_result.data, "git_status 返回结构异常")
ensure(len(git_commits_result.data["commits"]) <= 3, "git_recent_commits limit 未生效")
ensure("summary" in git_diff_result.data, "git_diff_summary 返回结构异常")
ensure(Path(tts_result.data["output_path"]).exists(), "text_to_speech 未写出音频文件")
ensure(Path(image_result.data["output_path"]).exists(), "generate_image 未写出图片文件")
ensure("analysis_text" in video_result.data, "analyze_video 返回结构异常")
ensure(Path(video_result.data["txt_output_path"]).exists(), "analyze_video txt 输出缺失")
ensure(Path(video_result.data["json_output_path"]).exists(), "analyze_video json 输出缺失")

dump_sample(
    {
        "custom_tools": [item["function"]["name"] for item in custom_registry.list_specs()],
        "filesystem_tools": [item["function"]["name"] for item in fs_registry.list_specs()],
        "execution_tools": [item["function"]["name"] for item in exec_registry.list_specs()],
        "git_tools": [item["function"]["name"] for item in git_registry.list_specs()],
        "media_tools": [item["function"]["name"] for item in media_registry.list_specs()],
        "run_command": command_result.data,
        "git_status": git_status_result.data,
        "tts": tts_result.data,
        "image": image_result.data,
        "video": video_result.data,
    }
)

await custom_registry.aclose()
await fs_registry.aclose()
await exec_registry.aclose()
await git_registry.aclose()
await media_registry.aclose()
await sandbox.aclose()
print("工具接口 smoke test 通过。")

In [ ]:
banner("Facade, Design, Workflow and Evolution APIs")

single_agent = agentorch.create_agent(
    model=DummyModel(reply="single-ok"),
    system_prompt="You are concise.",
    reasoning="react",
    enable_streaming=False,
)
single_result = await single_agent.run("say hello", thread_id="nb-single-001")
single_parsed = await single_agent.run_parsed("hello parsed", thread_id="nb-single-002", parser=agentorch.TextParser())
single_blueprint = single_agent.export_blueprint()

planner = agentorch.create_agent(model=DummyModel(name="planner-model", reply="plan ready"), reasoning="plan_execute", name="planner")
reviewer = agentorch.create_agent(model=DummyModel(name="reviewer-model", reply="review ready"), reasoning="react", name="reviewer")
team = agentorch.create_multi_agent(
    model=DummyModel(name="supervisor-model", reply="supervisor-ok"),
    agents=[
        {"agent": planner, "name": "planner", "role": "planner"},
        {"agent": reviewer, "name": "reviewer", "role": "reviewer"},
    ],
    system_prompt="Coordinate specialists and return one final answer.",
    name="demo-team",
)
team_result = await team.run("draft and review a plan", thread_id="nb-team-001")

design_agent = agentorch.AgentDesign.named("researcher", model=DummyModel(reply="design-ok"), profile="default").with_reasoning("react")
design_built = design_agent.build()
design_result = await design_built.run("design path", thread_id="nb-design-001")

composed_agent = agentorch.compose_agent({"name": "dict-agent", "model": DummyModel(reply="dict-ok")})
composed_result = await composed_agent.run("dict path", thread_id="nb-compose-001")

role_defaults = agentorch.AgentDesign(profile="default").with_tool_bundles(include_filesystem=True, include_git=True)
team_design = (
    agentorch.TeamDesign(name="delivery-team", role_defaults=role_defaults)
    .add_role(
        "planner",
        description="Planning lead",
        design={"model": DummyModel(name="role-planner", reply="role plan ready")},
        capabilities=["plan"],
        supports_parallel_tasks=True,
    )
)
team_design_built = team_design.build()
team_design_result = await team_design_built.run("plan this task", thread_id="nb-team-design-001")

skill_root = TMP_ROOT / "demo-skill"
skill_root.mkdir(exist_ok=True)
(skill_root / "SKILL.md").write_text(
    "---\nname: explicit-skill\ndescription: Explicit notebook skill\n---\nLoad the explicit notebook skill.",
    encoding="utf-8",
)
recording_model = RecordingPromptModel(reply="skill-ok")
skill_agent = agentorch.AgentDesign(
    model=recording_model,
    workspace_root=TMP_ROOT,
    skills=skill_root,
    skill_catalog={"discovery_roots": [".claude/skills"]},
    skill_routing=SkillRoutingConfig(mode="progressive", disclosure_level="progressive", selection_mode="model"),
).build()
skill_result = await skill_agent.run(
    "record the explicit skill prompt",
    thread_id="nb-skill-001",
    skill_request={"force_load": ["explicit-skill"], "allow_auto_select": False},
)

workflow = agentorch.Workflow.chain(
    Node.model_node("answer", prompt="answer the user"),
    Node.tool("lookup", "add_numbers", arguments={"a": 7, "b": 5}),
    entry_node="answer",
)
workflow_tools = agentorch.ToolRegistry.from_tools(add_numbers)
workflow_agent = agentorch.create_agent(model=DummyModel(reply="workflow-ok"), tools=workflow_tools, workflow=workflow)
workflow_result = await workflow_agent.run("workflow path", thread_id="nb-workflow-001")
workflow_payload = json.loads(workflow_result.output_text)

search_space = {"reasoning.kind": ["react"], "rag.mode": ["off"]}

async def agent_evaluator(genome, candidate, tasks):
    result = await candidate.run("solve task", thread_id=f"eval-agent-{genome.id}")
    return agentorch.EvaluationResult(
        genome_id=genome.id,
        fitness=1.0,
        metrics={"tasks": float(len(tasks)), "tokens": float(result.usage.total_tokens)},
    )


async def team_evaluator(genome, candidate, tasks):
    result = await candidate.run("coordinate task", thread_id=f"eval-team-{genome.id}")
    return agentorch.EvaluationResult(
        genome_id=genome.id,
        fitness=2.0,
        metrics={"tasks": float(len(tasks)), "tokens": float(result.usage.total_tokens)},
    )


agent_session = agentorch.create_agent_evolution(
    search_space=search_space,
    evolution_config=agentorch.EvolutionConfig(
        algorithm_kind="random_search",
        population_size=1,
        generations=1,
        evaluation_budget=1,
        seed=11,
    ),
    model=DummyModel(reply="evo-agent"),
    evaluator=agent_evaluator,
    tasks=["task-a"],
    name="evo-agent",
)
agent_evolution_result = await agent_session.evolve()
agent_candidate = await agent_session.build_best_candidate(agent_evolution_result)
agent_candidate_summary = await agent_session.summarize_candidate(agent_candidate)
await agent_candidate.aclose()

team_session = agentorch.create_multi_agent_evolution(
    search_space=search_space,
    evolution_config=agentorch.EvolutionConfig(
        algorithm_kind="random_search",
        population_size=1,
        generations=1,
        evaluation_budget=1,
        seed=13,
    ),
    roles=[
        {
            "name": "planner",
            "model": DummyModel(name="planner-model", reply="plan-ready"),
            "capabilities": ["plan"],
        }
    ],
    evaluator=team_evaluator,
    tasks=["task-b"],
    name="evo-team",
)
team_evolution_result = await team_session.evolve()
team_candidate = await team_session.build_best_candidate(team_evolution_result)
team_candidate_summary = await team_session.summarize_candidate(team_candidate)
await team_candidate.aclose()

ensure(single_result.output_text == "single-ok", "create_agent 运行失败")
ensure(single_parsed.parsed == "single-ok", "run_parsed 失败")
ensure(single_blueprint["kind"] == "single_agent", "single blueprint kind 异常")
ensure("plan ready" in team_result.output_text or "review ready" in team_result.output_text, "create_multi_agent 运行失败")
ensure(design_result.output_text == "design-ok", "AgentDesign.build 失败")
ensure(composed_result.output_text == "dict-ok", "compose_agent 失败")
ensure("role plan ready" in team_design_result.output_text, "TeamDesign.build 失败")
ensure(skill_result.output_text == "skill-ok", "skill agent 运行失败")
ensure(any("Load the explicit notebook skill." in prompt for prompt in recording_model.system_prompts), "显式 skill 未注入 system prompt")
ensure(workflow_payload["output"]["sum"] == 12, "workflow tool node 执行失败")
ensure(agent_evolution_result.best_evaluation.fitness == 1.0, "create_agent_evolution 失败")
ensure(agent_candidate_summary["facade"] == "create_agent", "agent evolution candidate summary 异常")
ensure(team_evolution_result.best_evaluation.fitness == 2.0, "create_multi_agent_evolution 失败")
ensure(team_candidate_summary["facade"] == "create_multi_agent", "team evolution candidate summary 异常")

dump_sample(
    {
        "single_blueprint": single_blueprint,
        "team_blueprint": team.export_blueprint(),
        "workflow_result": workflow_payload,
        "agent_evolution": agent_evolution_result.model_dump(mode="json"),
        "team_evolution": team_evolution_result.model_dump(mode="json"),
        "agent_candidate_summary": agent_candidate_summary,
        "team_candidate_summary": team_candidate_summary,
    }
)

for resource in [single_agent, team, planner, reviewer, design_built, composed_agent, team_design_built, skill_agent, workflow_agent]:
    await resource.aclose()
print("门面、设计、工作流、演化接口 smoke test 通过。")

In [ ]:
banner("Optional Live OpenAI-Compatible Smoke Test")

required_envs = [name for name in ["OPENAI_API_KEY"] if os.getenv(name)]
base_url = os.getenv("OPENAI_BASE_URL")
chat_model = os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL")

if required_envs and chat_model:
    print("检测到当前 shell 已存在可用环境变量，开始真实链路最小 smoke test。")
    live_agent = agentorch.create_agent(
        model=agentorch.OpenAIModel(api_key=os.getenv("OPENAI_API_KEY"), base_url=base_url, model=chat_model),
        system_prompt="Reply with exactly: live smoke ok",
        enable_streaming=False,
    )
    try:
        live_result = await live_agent.run("Reply with exactly: live smoke ok", thread_id="nb-live-001")
        dump_sample(
            {
                "base_url": base_url,
                "model": chat_model,
                "output_text": live_result.output_text,
                "usage": live_result.usage.model_dump(),
            }
        )
    finally:
        await live_agent.aclose()
else:
    print("跳过真实模型 smoke test：当前 shell 未同时提供 OPENAI_API_KEY 与可用模型名。")
    print("需要的最小环境变量：OPENAI_API_KEY，以及 OPENAI_CHAT_MODEL / OPENAI_MODEL / AGENTORCH_MODEL 之一。")

In [ ]:
banner("Notebook Summary")
summary = {
    "distribution_name": "masarch",
    "import_name": "agentorch",
    "stable_export_count": len(set(agentorch.STABLE_API_SYMBOLS)),
    "compat_export_count": len(set(agentorch.COMPAT_API_SYMBOLS)),
    "tmp_root": str(TMP_ROOT),
}
dump_sample(summary)
print("离线核心 smoke test 已完成。")